In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

page_visits_data = [
    ("U1", "P1", "2023-01-01 12:00:00"),
    ("U2", "P3", "2023-01-02 15:30:00"),
    ("U3", "P2", "2023-01-03 10:45:00"),
]
page_likes_data = [
    ("U1", "P2", "2023-01-02 14:20:00"),
    ("U2", "P1", "2023-01-03 16:40:00"),
    ("U3", "P3", "2023-01-04 18:55:00"),
]
page_comments_data = [
    ("U1", "P3", "2023-01-03 13:00:00"),
    ("U2", "P2", "2023-01-04 17:10:00"),
    ("U3", "P1", "2023-01-05 19:25:00"),
]

page_visits = spark.createDataFrame(
    page_visits_data, ["user_id", "page_id", "visit_time"]
).withColumn("visit_time", to_timestamp("visit_time"))
page_likes = spark.createDataFrame(
    page_likes_data, ["user_id", "page_id", "like_time"]
).withColumn("like_time", to_timestamp("like_time"))
page_comments = spark.createDataFrame(
    page_comments_data, ["user_id", "page_id", "comment_time"]
).withColumn("comment_time", to_timestamp("comment_time"))

# quick display
page_visits.show(truncate=False)
page_likes.show(truncate=False)
page_comments.show(truncate=False)

+-------+-------+-------------------+
|user_id|page_id|visit_time         |
+-------+-------+-------------------+
|U1     |P1     |2023-01-01 12:00:00|
|U2     |P3     |2023-01-02 15:30:00|
|U3     |P2     |2023-01-03 10:45:00|
+-------+-------+-------------------+

+-------+-------+-------------------+
|user_id|page_id|like_time          |
+-------+-------+-------------------+
|U1     |P2     |2023-01-02 14:20:00|
|U2     |P1     |2023-01-03 16:40:00|
|U3     |P3     |2023-01-04 18:55:00|
+-------+-------+-------------------+

+-------+-------+-------------------+
|user_id|page_id|comment_time       |
+-------+-------+-------------------+
|U1     |P3     |2023-01-03 13:00:00|
|U2     |P2     |2023-01-04 17:10:00|
|U3     |P1     |2023-01-05 19:25:00|
+-------+-------+-------------------+



In [4]:
page_visits = page_visits.withColumn("interaction_type", lit("visit"))
page_likes = page_likes.withColumn("interaction_type", lit("like"))
page_comments = page_comments.withColumn("interaction_type", lit("comment"))

In [6]:
page_visits.union(page_likes).union(page_comments).orderBy(col("visit_time")).show()

+-------+-------+-------------------+----------------+
|user_id|page_id|         visit_time|interaction_type|
+-------+-------+-------------------+----------------+
|     U1|     P1|2023-01-01 12:00:00|           visit|
|     U1|     P2|2023-01-02 14:20:00|            like|
|     U2|     P3|2023-01-02 15:30:00|           visit|
|     U3|     P2|2023-01-03 10:45:00|           visit|
|     U1|     P3|2023-01-03 13:00:00|         comment|
|     U2|     P1|2023-01-03 16:40:00|            like|
|     U2|     P2|2023-01-04 17:10:00|         comment|
|     U3|     P3|2023-01-04 18:55:00|            like|
|     U3|     P1|2023-01-05 19:25:00|         comment|
+-------+-------+-------------------+----------------+

